In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r"C:\Users\ADMIN\Desktop\TDI_Python\capstodata.csv")

df.head()

In [ ]:
df.isnull().sum()


In [ ]:
#This drop the 'Gendr.1' column with empty value

df.drop(columns=["Gender.1"], inplace=True)

In [ ]:
#To check again 

df.isnull().sum()

In [ ]:
df.info()

In [ ]:
#Adding a new column called 'Status' to know the employee active and inactive.

df["Status"] = np.where(df["Exit_Date"].isna(), "Active", "Inactive")

df.head()

In [ ]:
#I have 18 columns but this select only the on to be display

selected_columns = [
    "Employee_Id",
    "Full_Name",
    "Gender",
    "Age",
    "Education_Level",
    "Department",
    "Job_Role",
    "Job_Level",
    "Employment_Type",
    "Store_Location",
    "Base_Salary_Annual",
    "Status"
]

df[selected_columns].head()


In [ ]:
df['Full_Name'].value_counts()

In [ ]:
df['Gender'].value_counts()

In [ ]:
df['Age'].value_counts()

In [ ]:
df['Education_Level'].value_counts()

In [ ]:
df['Department'].value_counts()

In [ ]:
df['Employment_Type'].value_counts()

In [ ]:
df['Job_Level'].value_counts()

In [ ]:
df['Store_Location'].value_counts()

In [ ]:
#Validation of the status of my dataset 
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

In [ ]:
# Check missing values in all columns
missing_summary = df.isnull().sum()
print("Missing values per column:")
print(missing_summary)


In [ ]:
#check data type 

print(df.dtypes)


In [ ]:
#Changed the data type to a right format

df['Hire_Date'] = pd.to_datetime(df['Hire_Date'], errors='coerce', dayfirst=True)
df['Exit_Date'] = pd.to_datetime(df['Exit_Date'], errors='coerce', dayfirst=True)

print(df.dtypes)

In [ ]:
df.describe()


In [ ]:
"""
 RQ1) What are the demographic and job-level patterns of the workforce (age, gender, education)?
 """

# Age distribution
plt.figure(figsize=(10,4))
sns.histplot(df['Age'].dropna(), bins=20, kde=True, color='skyblue')
plt.title('Distribution of Employee Age')
plt.show()

# Gender count
plt.figure(figsize=(10,4))
sns.countplot(x='Gender', hue='Gender', data=df, palette='Set2', legend=True)
plt.title('Gender Distribution')
plt.show()

# Job level count
plt.figure(figsize=(10,4))
sns.countplot(x='Job_Level', hue='Job_Level', data=df, palette='pastel', legend=True)
plt.title('Job Level Distribution')
plt.show()

In [ ]:
"""
RQ2) What factors (department, job level, employment type) influence employee turnover (active vs inactive)?
"""

# Turnover rate by Department
turnover = df.groupby('Department')['Status'].value_counts().unstack(fill_value=0)
turnover['Turnover_Rate'] = turnover.get('Inactive', 0) / turnover.sum(axis=1)
turnover[['Turnover_Rate']].sort_values('Turnover_Rate', ascending=False).head(10)


In [ ]:
# Visualization: Turnover Rate by Department
plt.figure(figsize=(10,5))
turnover_sorted = turnover.sort_values('Turnover_Rate', ascending=False).head(10)
sns.barplot(x=turnover_sorted['Turnover_Rate'], y=turnover_sorted.index, palette='Reds_r' )
plt.title('Top 10 Departments with Highest Turnover Rates')
plt.xlabel('Turnover Rate')
plt.ylabel('Department')
plt.show()


In [ ]:
"""
RQ3) How do salary and tenure vary by department, job level, and employment type?
"""

# Average salary by Job Level
salary_by_level = df.groupby('Job_Level')['Base_Salary_Annual'].mean().sort_index()
salary_by_level.plot(kind='bar', color='purple', title='Average Salary by Job Level')
plt.ylabel('Average Salary')
plt.show()


In [ ]:
# Calculate tenure in years
df['Hire_Date'] = pd.to_datetime(df['Hire_Date'], errors='coerce', dayfirst=True)
df['Exit_Date'] = pd.to_datetime(df['Exit_Date'], errors='coerce', dayfirst=True)
df['Tenure_Years'] = ((df['Exit_Date'].fillna(pd.Timestamp.today()) - df['Hire_Date']).dt.days / 365).round(1)


# Count active vs inactive per location
summary = df.groupby(['Store_Location', 'Status']).size().unstack(fill_value=0)
summary['Total'] = summary.sum(axis=1)
summary['Turnover_Rate'] = summary.get('Inactive', 0) / summary['Total']

# Add average tenure
avg_tenure = df.groupby('Store_Location')['Tenure_Years'].mean()
summary['Avg_Tenure'] = avg_tenure
summary = summary.reset_index().sort_values('Turnover_Rate', ascending=False)

summary.head(10)


In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=summary, x='Avg_Tenure', y='Turnover_Rate', hue='Turnover_Rate', palette='coolwarm')
plt.title('Tenure vs Turnover Rate by Location')
plt.xlabel('Average Tenure (Years)')
plt.ylabel('Turnover Rate')
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
dept_salary = df.groupby('Department')['Base_Salary_Annual'].mean().sort_values(ascending=False).head(10)
sns.barplot(x=dept_salary.values, y=dept_salary.index, palette='viridis')
plt.title('Average Salary by Department (Top 10)')
plt.xlabel('Average Annual Salary ($)')
plt.ylabel('Department')
plt.show()



In [ ]:
"""
RQ4) How do age, job level, and salary correlate within the organization?
"""
 

 #  Map job levels to numeric values
df["Job_Level_Num"] = df["Job_Level"].map({
    "Entry": 1,
    "Mid": 2,
    "Senior": 3,
    "Manager": 4,
    "Director": 5
})

# clean Base_Salary_Annual (remove $ or commas, if any)
df["Base_Salary_Annual"] = (
    df["Base_Salary_Annual"]
    .replace('[\$,]', '', regex=True)
    .astype(float)
)

#  Now run correlation
corr_cols = ["Age", "Job_Level_Num", "Base_Salary_Annual"]
corr_matrix = df[corr_cols].corr()

#  Visualize
plt.figure(figsize=(6,4))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation: Age, Job Level, and Salary")
plt.show()



In [ ]:
'''
RQ5) How do location and tenure trends influence workforce distribution and retention? 
'''
# Calculate tenure (in years)
df['Tenure'] = (df['Exit_Date'].fillna(pd.Timestamp.today()) - df['Hire_Date']).dt.days / 365

# Group by store/location
tenure_by_location = df.groupby('Store_Location')['Tenure'].mean().sort_values(ascending=False).head(10)

# Visualize
plt.figure(figsize=(10,5))
tenure_by_location.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Average Employee Tenure by Top 10 Store Locations')
plt.ylabel('Average Tenure (Years)')
plt.xlabel('Store Location')
plt.xticks(rotation=45)
plt.show()
